In [0]:
from pyspark.sql import functions as F

bronze = spark.table("workspace.default.aqi_bronze")
print("Bronze rows:", bronze.count())

Bronze rows: 29531


In [0]:
silver = (
    bronze
    .withColumn("City", F.trim(F.col("City")))
    .withColumn("Date", F.to_date(F.col("Date")))
    .withColumn("Year", F.year("Date"))
    .withColumn("Month", F.month("Date"))
    .withColumn(
        "Season",
        F.when(F.col("Month").isin(12, 1, 2), "Winter")
         .when(F.col("Month").isin(3, 4, 5), "Summer")
         .when(F.col("Month").isin(6, 7, 8, 9), "Monsoon")
         .otherwise("Post-Monsoon")
    )
)

print("Rows:", silver.count())
display(silver.select("City", "Date", "Year", "Month", "Season", "AQI").limit(5))

Rows: 29531


City,Date,Year,Month,Season,AQI
Ahmedabad,2015-01-01,2015,1,Winter,null
Ahmedabad,2015-01-02,2015,1,Winter,null
Ahmedabad,2015-01-03,2015,1,Winter,null
Ahmedabad,2015-01-04,2015,1,Winter,null
Ahmedabad,2015-01-05,2015,1,Winter,null


In [0]:
pollutants = ["PM2_5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3", "Benzene", "Toluene", "Xylene"]

all_null = F.lit(True)
for c in pollutants + ["AQI"]:
    all_null = all_null & F.col(c).isNull()

before = silver.count()
silver = silver.filter(~all_null).drop("Xylene")
after = silver.count()

print("Deleting rows:", before - after)
print("Remaning rows:", after)

Deleting rows: 1374
Remaning rows: 28157


In [0]:
print("Rows:", silver.count())
print("Xylene present:", "Xylene" in silver.columns)
print("Columns:", len(silver.columns))

Rows: 28157
Xylene present: False
Columns: 18


In [0]:
silver = silver.withColumn(
    "AQI_Bucket",
    F.when(F.col("AQI").isNull(), None)
     .when(F.col("AQI") <= 50, "Good")
     .when(F.col("AQI") <= 100, "Satisfactory")
     .when(F.col("AQI") <= 200, "Moderate")
     .when(F.col("AQI") <= 300, "Poor")
     .when(F.col("AQI") <= 400, "Very Poor")
     .otherwise("Severe")
)

display(silver.groupBy("AQI_Bucket").count().orderBy("count", ascending=False))

AQI_Bucket,count
Moderate,8829
Satisfactory,8224
null,3307
Poor,2781
Very Poor,2337
Good,1341
Severe,1338


In [0]:
silver = silver.withColumn("AQI_Above_500", F.col("AQI") > 500)

In [0]:
silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.aqi_silver")

print("Rows:", spark.table("workspace.default.aqi_silver").count())
print("Columns:", len(spark.table("workspace.default.aqi_silver").columns))
print("Rows above 500:", spark.table("workspace.default.aqi_silver").filter(F.col("AQI_Above_500") == True).count())

Rows: 28157
Columns: 19
Rows above 500: 543
